# Benson Increments Calculator

### Chemistry C450/C540, Indiana University Bloomington &mdash; Prashant Kumar and Dr. Nicola L. B. Pohl

**Generation 4** &mdash; 2026-09-15 version.

This notebook estimates the standard enthalpy of formation ($\Delta H_f^\circ$)
of an organic molecule from Benson group increments (Cohen & Benson, *Chem.
Rev.* **1993**, *93*, 2419, and other tables &mdash; see the repository
[README](https://github.com/PrashantKumarChem/Benson-Increments-Calculator#readme)
for provenance). You choose which increments and corrections your molecule
needs; the notebook totals them and shows the method's own uncertainty
alongside the total. It does not perceive molecular structure &mdash; that
withholding is the pedagogical point, not a missing feature.

**What changed from the previous notebook.** This generation imports
[`benson`](https://github.com/PrashantKumarChem/Benson-Increments-Calculator/tree/main/benson),
the same pure-Python package the website's data is built from, instead of
carrying its own copy of the value-parsing and notation rules. There is one
implementation of the chemistry rules in this project, and this notebook is a
consumer of it, like the website. It also **runs in Colab with nothing cloned
locally** &mdash; the cell below installs the package from GitHub, and the one
after fetches the curated data over HTTPS.

The earlier, self-contained notebook (its own CSV parser, no package) is
archived at
[`Archives/gen2-Benson Increments Calculator.ipynb`](https://github.com/PrashantKumarChem/Benson-Increments-Calculator/blob/main/Archives/gen2-Benson%20Increments%20Calculator.ipynb),
frozen with its own copy of the data it read.

**Scope.** This notebook covers browsing by category, searching by notation
shorthand, adding/removing/counting increments, the running total with its
mixed-quantity warning and uncertainty line, copying your working-out as
plain text, and bringing your own data. Things that are presentation-only on
the [website](https://prashantkumarchem.github.io/Benson-Increments-Calculator/)
&mdash; card/table toggle, the phone slide-up sheet, keyboard grid navigation,
theming, responsive layout &mdash; stay website-only; they would just be a
second, thinner copy of the same UI here.


## 1. Install `benson`

Pure Python, zero runtime dependencies (that is deliberate &mdash; see the
package's own docstring) &mdash; so this is instant, in Colab and locally.
Installs from the repository's default branch, since this notebook is a
consumer of the package and does not change what it does.

In [ ]:
%pip install -q git+https://github.com/PrashantKumarChem/Benson-Increments-Calculator.git

## 2. Import `benson`, and a small HTTPS fetch helper

Colab has no local clone, so the curated data and the notation tables (what
`benson.notation.load_notation()` needs to read a Benson group name) are
fetched over HTTPS rather than found on disk. `fetch()` is the only reader of
a URL in this notebook; everything after this cell reads what it wrote to a
local file, the same way it would from a clone.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import json
import urllib.request

import benson
from benson.build import build_artifact
from benson.notation import normalise, score_of

print("benson", benson.__version__, "from", benson.__file__)

REPO_RAW = "https://raw.githubusercontent.com/PrashantKumarChem/Benson-Increments-Calculator/main"


def fetch(path):
    # One file from the repository's default branch, as text.
    with urllib.request.urlopen(f"{REPO_RAW}/{path}") as response:
        return response.read().decode("utf-8")


def fetch_json(path):
    return json.loads(fetch(path))


## 3. The curated data

`dist/increments.json` is the artifact `benson/build.py` derives from
`CSV_data_files/`, `notation/` and the reference tables. It already carries
every increment's value, its notation decomposition and its precomputed
search aliases &mdash; this notebook does no parsing of its own, the same way
the website does none.

In [ ]:
ARTIFACT = fetch_json("dist/increments.json")
print(len(ARTIFACT["increments"]), "increments across", len(ARTIFACT["categories"]), "categories")
CATEGORIES_BY_FILE = {c["file"]: c for c in ARTIFACT["categories"]}


## 4. The rules that stay written twice

A generated artifact removes drift from the data and everything derived from
it, but not from the handful of small rules every *consumer* still applies at
selection time &mdash; how a running total is summed, how it is formatted, and
that the method's own uncertainty is never built up from the individual
groups' own (D11: ten groups at &plusmn;3 kJ/mol is not &plusmn;9.5). The
website carries these in `assets/format.js`; this notebook is a second
consumer, so it carries the same small rules again, deliberately kept short
enough to read in one screen. The next cell checks this copy against the
project's own conformance fixture rather than trusting it by inspection.

In [ ]:
# entries: [(value, count), ...] -> sum(value * count).
def tally(entries):
    return sum(value * count for value, count in entries)


# One decimal place; an explicit sign; zero takes none - assets/format.js's formatTotal().
def format_total(kj):
    text = f"{abs(kj):.1f}"
    if kj > 0:
        return f"+{text}"
    if kj < 0:
        return f"\u2212{text}"
    return text


# How far the total could move if every chosen published range were read at its bounds.
def range_spread(entries):
    return sum(((e["high"] - e["low"]) / 2) * count for e, count in entries if e["isRange"])


# The method's own published figure, never a quadrature sum of the chosen
# groups' own uncertainties (D11) - individual_uncertainties is accepted and
# deliberately unused, the same signature assets/format.js's does.
def combined_uncertainty(individual_uncertainties, method_figure):
    return method_figure


# entries: [(increment dict, count), ...] -> what the sum is a total *of*.
# Categories do not all hold the same physical quantity - a Benson group
# increment is an enthalpy of formation, a cyclohexane A-value is a
# conformational free energy. Both are summed, but a heading claiming one
# quantity over a total that mixes them would be wrong, so a mixed selection
# is labelled plainly instead of guessed at.
def describe_total(entries):
    fallback = {"label": "Total", "quantities": [], "mixed": False}
    if not entries:
        return fallback
    seen = {}
    for e, count in entries:
        category = CATEGORIES_BY_FILE.get(e["category"])
        if not category or not category.get("symbol") or not category.get("quantity"):
            return fallback
        q = seen.setdefault(category["symbol"], dict(category, count=0))
        q["count"] += count
    quantities = list(seen.values())
    if len(quantities) == 1:
        return {"label": quantities[0]["symbol"], "quantities": quantities, "mixed": False}
    return {"label": "Total", "quantities": quantities, "mixed": True}


ELEMENT_NAMES = {"C": "carbon", "H": "hydrogen", "N": "nitrogen", "O": "oxygen"}


def _or_list(words):
    words = list(words)
    return f"{', '.join(words[:-1])} or {words[-1]}" if len(words) > 1 else "".join(words)


# What to say about the method's own error under the total, or None. Mirrors
# assets/format.js's describeMethodUncertainty(): the published figure is
# shown as-is (never rebuilt from the chosen groups' own uncertainties -
# combined_uncertainty() is what says so), and only for a quantity the
# selection actually contains a figure for.
def describe_method_uncertainty(entries):
    figures = ARTIFACT["uncertainty"]
    if not figures:
        return None
    described = describe_total(entries)
    if not described["quantities"]:
        return None

    with_figure = [(q, next((f for f in figures if f["symbol"] == q["symbol"]), None))
                   for q in described["quantities"]]
    with_figure = [(q, f) for q, f in with_figure if f is not None]
    if not with_figure:
        return [f"No method error is given for a "
                f"{_or_list(q['symbol'] for q in described['quantities'])} total; "
                f"the figure is for {_or_list(f['symbol'] for f in figures)}."]

    sentences = []
    for quantity, figure in with_figure:
        terms = [(e, c) for e, c in entries
                 if CATEGORIES_BY_FILE.get(e["category"], {}).get("symbol") == quantity["symbol"]]
        own = [e["uncertainty"] for e, c in terms for _ in range(c) if e["uncertainty"] is not None]
        value = combined_uncertainty(own, figure["value"])
        sentences.append(f"Benson estimates of {figure['phase']}-phase {figure['symbol']} are "
                          f"typically off by about {value:.{figure['decimals']}f} kJ/mol.")
        if described["mixed"]:
            sentences.append(f"The figure is for the {figure['symbol']} terms only.")
        outside = sorted({el for e, c in terms for el in e["composition"].keys()} - set(figure["elements"]))
        if outside:
            sentences.append("The figure does not cover " +
                              _or_list(ELEMENT_NAMES.get(el, el) for el in outside) + ".")
    return sentences


# The chosen increments as plain text, for pasting into a report.
def format_selection_as_text(entries, total_kj, described):
    if not entries:
        return ""
    names = [e["label"] + (f" x{c}" if c > 1 else "") for e, c in entries]
    sums = [format_total(e["value"] * c) for e, c in entries]
    name_w = max(max(len(n) for n in names), 8)
    sum_w = max(max(len(s) for s in sums), 8)
    lines = [f"{n:<{name_w}}  {s:>{sum_w}}" for n, s in zip(names, sums)]
    lines.append("-" * (name_w + 2 + sum_w))
    lines.append(f"{described['label']:<{name_w}}  {format_total(total_kj):>{sum_w}} kJ/mol")
    kcal = total_kj * ARTIFACT["display"]["kj_to_kcal"]
    lines.append(f"{'':<{name_w}}  {format_total(kcal):>{sum_w}} kcal/mol")
    if described["mixed"]:
        lines.append("")
        lines.append("Mixes " + ", ".join(f"{q['count']} x {q['quantity']}" for q in described["quantities"]) + ".")
    return "\n".join(lines)


## 5. Checking that copy against the project's own fixture

`tests/conformance.json` is the input&rarr;expected-output contract both the
website's JavaScript and the `benson` package's own Python test suite are
held to (D7 layer C) &mdash; it exists precisely because this handful of rules
stays written more than once. This notebook is a third copy, so it is checked
against the same fixture rather than trusted by inspection. The `query` cases
run against `benson.notation.score_of()` directly &mdash; that part is real
package code, not a copy.

In [ ]:
CONFORMANCE = fetch_json("tests/conformance.json")

_checked = 0
for case in CONFORMANCE["cases"]:
    rule, inp, expect = case["rule"], case["input"], case["expect"]
    if rule == "tally":
        got = tally([(e["value"], e["count"]) for e in inp["entries"]])
    elif rule == "total":
        got = format_total(inp["kj"])
    elif rule == "uncertainty":
        got = combined_uncertainty(inp["individualUncertainties"], inp["methodFigure"])
    elif rule == "query":
        match = score_of(inp["query"], inp["aliases"])
        got = match.name.lower() if match is not None else None
    else:
        continue
    assert got == expect, f"{case['id']}: got {got!r}, expected {expect!r} ({case['why']})"
    _checked += 1

print(f"OK - {_checked} conformance cases agree (tally, total, uncertainty, query).")


## 6. Browse, search, select

Buttons are grouped by category, as the CSV files are ordered. Search reads
the same precomputed aliases the website's search does &mdash; `CH3` finds
`C-(C)(H)3` the same way here. Add, remove and adjust a count; the running
total, its mixed-quantity warning and the method's own uncertainty line
update together; copy your working-out from the text box at the bottom.

In [ ]:
selection = {}  # label -> count
INCREMENTS_BY_LABEL = {e["label"]: e for e in ARTIFACT["increments"]}

total_label = widgets.HTML()
mixed_label = widgets.HTML()
uncertainty_label = widgets.HTML()
selection_box = widgets.VBox()
search_box = widgets.Text(placeholder="Search by name or shorthand, e.g. CH3")
search_results = widgets.VBox()
working_out = widgets.Textarea(layout=widgets.Layout(width="480px", height="140px"), disabled=True)


def selected_entries():
    return [(INCREMENTS_BY_LABEL[label], count) for label, count in selection.items() if count > 0]


def add(label, delta=1):
    selection[label] = max(0, selection.get(label, 0) + delta)
    if selection[label] == 0:
        del selection[label]
    refresh()


def refresh():
    entries = selected_entries()
    total_kj = tally([(e["value"], c) for e, c in entries])
    described = describe_total(entries)
    total_label.value = (f"<b>{described['label']}:</b> {format_total(total_kj)} kJ/mol "
                          f"({format_total(total_kj * ARTIFACT['display']['kj_to_kcal'])} kcal/mol)")

    spread = range_spread(entries)
    spread_text = f" &plusmn;{spread:.2f} kJ/mol from the published ranges chosen" if spread else ""
    mixed_text = ""
    if described["mixed"]:
        mixed_text = ("This total mixes " +
                       ", ".join(f"{q['count']} &times; {q['quantity']}" for q in described["quantities"]) +
                       ". They are summed as published.")
    mixed_label.value = mixed_text + spread_text

    sentences = describe_method_uncertainty(entries)
    uncertainty_label.value = "<br>".join(sentences) if sentences else ""

    rows = []
    for e, count in entries:
        label = e["label"]
        name = widgets.Label(f"{label}: {format_total(e['value'] * count)} kJ/mol",
                              layout=widgets.Layout(width="260px"))
        minus = widgets.Button(description="-", layout=widgets.Layout(width="32px"))
        count_label = widgets.Label(str(count), layout=widgets.Layout(width="24px"))
        plus = widgets.Button(description="+", layout=widgets.Layout(width="32px"))
        remove = widgets.Button(description="Remove", layout=widgets.Layout(width="70px"))
        minus.on_click(lambda b, l=label: add(l, -1))
        plus.on_click(lambda b, l=label: add(l, 1))
        remove.on_click(lambda b, l=label: add(l, -selection.get(l, 0)))
        rows.append(widgets.HBox([name, minus, count_label, plus, remove]))
    selection_box.children = rows or [widgets.HTML("<i>Nothing selected yet.</i>")]
    working_out.value = format_selection_as_text(entries, total_kj, described)


def category_grid(category):
    buttons = []
    for e in ARTIFACT["increments"]:
        if e["category"] != category["file"]:
            continue
        btn = widgets.Button(description=f"{e['label']}\n{format_total(e['value'])}",
                              layout=widgets.Layout(width="150px", height="40px"))
        btn.on_click(lambda b, l=e["label"]: add(l, 1))
        buttons.append(btn)
    return widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat(4, 150px)", grid_gap="2px"))


tab = widgets.Tab()
tab.children = [category_grid(c) for c in ARTIFACT["categories"]]
for i, c in enumerate(ARTIFACT["categories"]):
    tab.set_title(i, c["title"])


def on_search(change):
    query = change["new"]
    if not normalise(query):
        search_results.children = []
        return
    matches = []
    for e in ARTIFACT["increments"]:
        match = score_of(query, e["aliases"])
        if match is not None:
            matches.append((match, e))
    matches.sort(key=lambda pair: pair[0])
    buttons = []
    for match, e in matches[:24]:
        btn = widgets.Button(description=f"{e['label']} ({format_total(e['value'])})",
                              layout=widgets.Layout(width="220px"))
        btn.on_click(lambda b, l=e["label"]: add(l, 1))
        buttons.append(btn)
    search_results.children = buttons


search_box.observe(on_search, names="value")

refresh()
display(widgets.HTML("<h3>Browse by category</h3>"), tab)
display(widgets.HTML("<h3>Search</h3>"), search_box, search_results)
display(widgets.HTML("<h3>Selected</h3>"), selection_box)
display(total_label, mixed_label, uncertainty_label)
display(widgets.HTML("<h3>Copy your working-out</h3>"), working_out)


## 7. Bring your own data

Point this at a folder of your own CSV files &mdash; two columns, a group name
and a value, exactly the format `CSV_data_files/` uses &mdash; and get back
**the same artifact shape** the curated data above uses: the same keys, read
the same way, because it is built by the same function,
`benson.build.build_artifact()`. Nothing here parses your file a second way.

This needs a `notation/` directory too &mdash; the tables that say what a
Benson group name like `C-(C)(H)3` is made of, so your own groups (if they
follow the same notation) get the same composition reading and search
aliases the curated groups do. The cell below fetches the project's own
tables; supply your own `notation/` folder instead if your labels use a
different convention.

In Colab, upload your CSV(s) to a folder (the file browser in the sidebar, or
`from google.colab import files; files.upload()`) and set `MY_DATA_DIR`
below.

In [ ]:
import os

NOTATION_FILES = ("central_atoms.csv", "ligand_atoms.csv", "special_labels.csv", "synonyms.csv")
os.makedirs("notation", exist_ok=True)
for name in NOTATION_FILES:
    with open(os.path.join("notation", name), "w", encoding="utf-8", newline="\n") as handle:
        handle.write(fetch(f"notation/{name}"))

# Point this at your own folder of CSV files. Uncomment to try it - the
# artifact you get back should look exactly like ARTIFACT above (same keys,
# same shape).
#
# MY_DATA_DIR = "my_data"
# my_artifact = build_artifact(csv_dir=MY_DATA_DIR, notation_dir="notation")
# print(sorted(my_artifact.keys()) == sorted(ARTIFACT.keys()))
# print(len(my_artifact["increments"]), "increments read from your own data")
#
# Reuse everything above on your own data by pointing ARTIFACT at it and
# re-running from cell 6:
#
# ARTIFACT = my_artifact
# CATEGORIES_BY_FILE = {c["file"]: c for c in ARTIFACT["categories"]}
